In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy import signal as sp_signal
from joblib import Parallel, delayed
from google.colab import drive

# 1. SETUP
drive.mount('/content/drive', force_remount=True)
DATA_ROOT = "/content/drive/MyDrive/capture24"
# New folder for research-grade data
OUT_DIR = os.path.join(DATA_ROOT, "processed_research")
ANNOTATION_PATH = os.path.join(DATA_ROOT, "annotation-label-dictionary.csv")
os.makedirs(OUT_DIR, exist_ok=True)

# 2. CONFIGURATION (Addressed Concern: Window Size)
FS = 100.0
WINDOW_SEC = 10.0  # Increased from 4s to 10s (Professor's request)
GRAVITY_WIN_SEC = 10
LOWPASS_CUTOFF = 20.0

# 3. LOAD LABEL DICTIONARY (Address Concern #2)
print("⏳ Loading Label Dictionary...")
try:
    annot_df = pd.read_csv(ANNOTATION_PATH)
    # Create mapping: "7030 sleeping" -> "sleep"
    # We use 'label:Willetts2018' as the standard grouping
    LABEL_MAP = dict(zip(annot_df['annotation'], annot_df['label:Willetts2018']))

    # Define our target numeric classes
    CLASSES = ['sleep', 'sit-stand', 'walking', 'mixed', 'bicycling', 'vehicle', 'unknown']
    CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
    print(f"✅ Loaded Dictionary. Target Classes: {CLASSES}")
except Exception as e:
    raise ValueError(f"❌ Could not load annotation file! Make sure {ANNOTATION_PATH} exists.\nError: {e}")

# 4. ROBUST FILE PROCESSOR
def process_file_research(path):
    try:
        out_name = os.path.basename(path).replace(".csv.gz", "_research.npz")
        out_path = os.path.join(OUT_DIR, out_name)

        if os.path.exists(out_path): return "SKIPPED"

        # Load CSV
        df = pd.read_csv(path, compression="gzip")

        # Identify Columns (Robust)
        t_col = next((c for c in df.columns if 'time' in c or 'ts' in c), None)
        a_cols = [c for c in df.columns if 'acc' in c or c in ['x','y','z']][:3]
        lbl_col = next((c for c in df.columns if 'annotation' in c), None) # Find label col

        if not t_col or len(a_cols) < 3: return "FAIL_COLS"

        # Extract Time & Accel
        t = pd.to_datetime(df[t_col], utc=True).astype('int64').values / 1e9
        X = df[a_cols].values.astype(float)

        # Resample to fixed 100Hz grid
        t0, t1 = t[0], t[-1]
        n_samples = int((t1 - t0) * FS)
        t_u = np.linspace(t0, t1, n_samples)

        Xu = np.zeros((n_samples, 3))
        for i in range(3):
            Xu[:, i] = interp1d(t, X[:, i], bounds_error=False, fill_value="extrapolate")(t_u)

        # Filtering (Gravity + Lowpass)
        win = int(GRAVITY_WIN_SEC * FS)
        Xc = Xu - pd.DataFrame(Xu).rolling(win, center=True, min_periods=1).mean().values
        b, a = sp_signal.butter(4, LOWPASS_CUTOFF/(FS/2), btype='low')
        Xf = sp_signal.filtfilt(b, a, Xc, axis=0)

        # PROCESS LABELS
        if lbl_col:
            raw_labels = df[lbl_col].astype(str).values
            # 1. Map raw string to standardized class string
            mapped_labels = np.array([LABEL_MAP.get(l, 'unknown') for l in raw_labels])
            # 2. Map class string to integer ID
            y_ids = np.array([CLASS_TO_ID.get(l, CLASS_TO_ID['unknown']) for l in mapped_labels])

            # Resample labels (Nearest Neighbor)
            f_lbl = interp1d(t, y_ids, kind='nearest', bounds_error=False, fill_value="extrapolate")
            Yu = f_lbl(t_u).astype(int)
        else:
            Yu = np.full(n_samples, CLASS_TO_ID['unknown'])

        # Windowing (10 seconds)
        W = int(WINDOW_SEC * FS)
        n_wins = len(Xf) // W
        if n_wins < 1: return "FAIL_LEN"

        X_wins = Xf[:n_wins*W].reshape(n_wins, W, 3)
        Y_wins = Yu[:n_wins*W].reshape(n_wins, W)

        # Majority Vote for Label per Window
        Y_final = np.array([np.bincount(w).argmax() for w in Y_wins])

        # Save X and Y
        np.savez_compressed(out_path, x=X_wins, y=Y_final)
        return "SUCCESS"

    except Exception as e:
        return f"ERROR: {str(e)}"

# RUN PROCESSING (Parallel)
raw_files = sorted(glob.glob(os.path.join(DATA_ROOT, "*.csv.gz")))
print(f"🚀 Processing {len(raw_files)} files with REAL LABELS...")
results = Parallel(n_jobs=4, verbose=5)(delayed(process_file_research)(f) for f in raw_files)
print(f"Done. Success: {results.count('SUCCESS')}")

Mounted at /content/drive
⏳ Loading Label Dictionary...
✅ Loaded Dictionary. Target Classes: ['sleep', 'sit-stand', 'walking', 'mixed', 'bicycling', 'vehicle', 'unknown']
🚀 Processing 151 files with REAL LABELS...


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:  2.6min
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed: 11.8min


Done. Success: 151


[Parallel(n_jobs=4)]: Done 151 out of 151 | elapsed: 27.1min finished


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
import numpy as np
import glob
import os
import matplotlib.pyplot as plt

# 1. CONFIGURATION
PROCESSED_DIR = "/content/drive/MyDrive/capture24/processed_research"
DEVICE = torch.device("cuda")
BATCH_SIZE = 256 # A100 allows large batches
PATCH_SIZE = 10  # Reduced to 0.1s (Address: "20 too coarse")
WINDOW_LEN = 1000 # 10s * 100Hz
PATCH_DIM = 3 * PATCH_SIZE # 30
NUM_PATCHES = WINDOW_LEN // PATCH_SIZE # 100 patches
D_MODEL = 256
N_HEAD = 8
NUM_LAYERS = 6
NUM_CLASSES = 7 # Matches the classes in Part 1

# 2. AUGMENTATIONS (Address: Point 1)
class Augmentations:
    @staticmethod
    def jitter(x, sigma=0.05):
        return x + torch.randn_like(x) * sigma

    @staticmethod
    def scale(x, sigma=0.1):
        return x * (1.0 + torch.randn(1).to(x.device) * sigma)

    @staticmethod
    def rotate(x):
        # Random rotation around Z-axis (gravity)
        theta = (torch.rand(1) - 0.5) * np.pi / 2 # +/- 45 deg
        c, s = torch.cos(theta), torch.sin(theta)
        # Assuming input (N, 3)
        # Rotation matrix logic simplified for tensor broadcasting
        x_new = x.clone()
        x_new[..., 0] = x[..., 0]*c - x[..., 1]*s
        x_new[..., 1] = x[..., 0]*s + x[..., 1]*c
        return x_new

    @staticmethod
    def time_warp(x, warps=2):
        # Simple simulation: drop random indices and interpolate (resample)
        # For simplicity in tensor ops, we skip complex warping here to keep speed high
        return x # Placeholder for complex warp

# 3. DATASET & SPLITTING (Address: Point 4)
class MultiTaskDataset(Dataset):
    def __init__(self, file_paths, mode='train'):
        self.mode = mode
        self.data_x = []
        self.data_y = []

        # Load all into RAM (A100 has 80GB+, system has 160GB+)
        print(f"   Loading {len(file_paths)} files for {mode}...")
        for f in file_paths:
            try:
                d = np.load(f)
                x, y = d['x'], d['y'] # x shape (N, 1000, 3)

                # Patchify here to save RAM during training
                # (N, 1000, 3) -> (N, 100, 30)
                N = x.shape[0]
                x_patches = x.reshape(N, NUM_PATCHES, PATCH_DIM)

                self.data_x.append(x_patches)
                self.data_y.append(y)
            except: pass

        self.data_x = np.concatenate(self.data_x, axis=0)
        self.data_y = np.concatenate(self.data_y, axis=0)

        # Convert to Tensor
        self.data_x = torch.tensor(self.data_x, dtype=torch.float32)
        self.data_y = torch.tensor(self.data_y, dtype=torch.long)

    def __len__(self): return len(self.data_x)

    def __getitem__(self, idx):
        x = self.data_x[idx] # (100, 30)
        y = self.data_y[idx]

        if self.mode == 'train':
            # Reshape for augmentation (Time, 3)
            x_flat = x.view(-1, 3)
            # Apply Augmentations randomly
            if torch.rand(1) < 0.5: x_flat = Augmentations.jitter(x_flat)
            if torch.rand(1) < 0.5: x_flat = Augmentations.scale(x_flat)
            if torch.rand(1) < 0.3: x_flat = Augmentations.rotate(x_flat)
            x = x_flat.view(NUM_PATCHES, PATCH_DIM)

        # SSL Mask (15%)
        ssl_mask = torch.rand(NUM_PATCHES) < 0.15

        # Forecast Mask (Last 10%)
        fc_mask = torch.zeros(NUM_PATCHES, dtype=torch.bool)
        fc_mask[-10:] = True # Predict last 1 second (10 patches)

        return x, ssl_mask, fc_mask, y

# 4. MULTI-TASK MODEL (Address: Point 6)
class ResearchTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Linear(PATCH_DIM, D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, NUM_PATCHES, D_MODEL) * 0.02)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(D_MODEL, N_HEAD, 1024, 0.1, batch_first=True),
            NUM_LAYERS
        )
        # Task Heads
        self.head_ssl = nn.Linear(D_MODEL, PATCH_DIM) # Reconstruct
        self.head_har = nn.Linear(D_MODEL, NUM_CLASSES) # Classify

    def forward(self, x):
        # x: (B, T, F)
        B, T, _ = x.shape
        h = self.emb(x) + self.pos
        h = self.encoder(h)

        # SSL Output
        recon = self.head_ssl(h)

        # HAR Output (Global Average Pool)
        gap = h.mean(dim=1)
        logits = self.head_har(gap)

        return recon, logits

# 5. EXECUTION
if __name__ == "__main__":
    # Get Participant Splits (Point 4)
    all_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, "*.npz")))
    pids = sorted(list(set([os.path.basename(f).split('_')[0] for f in all_files])))

    split = int(0.8 * len(pids))
    train_pids = set(pids[:split])
    test_pids = set(pids[split:])

    train_files = [f for f in all_files if os.path.basename(f).split('_')[0] in train_pids]
    test_files = [f for f in all_files if os.path.basename(f).split('_')[0] in test_pids]

    print(f"🚀 Splitting by Participant: {len(train_pids)} Train / {len(test_pids)} Test")

    # Load Data
    train_ds = MultiTaskDataset(train_files, mode='train')
    test_ds = MultiTaskDataset(test_files, mode='test')

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    # Model Setup
    model = ResearchTransformer().to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=1e-4)
    crit_mse = nn.MSELoss()
    crit_ce = nn.CrossEntropyLoss()

    # TRAINING LOOP (Multi-Task)
    print("\n--- Starting Multi-Task Training ---")
    for epoch in range(3):
        model.train()
        l_sum = 0
        for x, m_ssl, m_fc, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            m_ssl = m_ssl.to(DEVICE)

            # Mask Input
            x_in = x.clone()
            x_in[m_ssl] = 0 # Mask for SSL

            # Forward
            recon, logits = model(x_in)

            # Losses
            l_ssl = crit_mse(recon[m_ssl], x[m_ssl])
            l_har = crit_ce(logits, y)

            # Joint Loss (Weighted)
            loss = l_ssl + 0.5 * l_har

            opt.zero_grad()
            loss.backward()
            opt.step()
            l_sum += loss.item()

        print(f"Epoch {epoch+1} Loss: {l_sum/len(train_loader):.4f}")

    # EVALUATION (Address: Point 3 & 5)
    print("\n--- Final Evaluation ---")
    model.eval()
    y_true, y_pred = [], []
    fc_mae, fc_rmse = [], []

    with torch.no_grad():
        for x, _, m_fc, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            m_fc = m_fc.to(DEVICE)

            # Forecast Mode: Mask future
            x_in = x.clone()
            x_in[m_fc] = 0

            recon, logits = model(x_in)

            # HAR Metrics
            y_true.extend(y.cpu().numpy())
            y_pred.extend(logits.argmax(1).cpu().numpy())

            # Forecast Metrics (Unscaled)
            err = (recon - x)[m_fc]
            fc_mae.append(torch.abs(err).mean().item())
            fc_rmse.append(torch.sqrt(torch.mean(err**2)).item())

    print(f"1. Forecasting (1.0s Horizon):")
    print(f"   MAE: {np.mean(fc_mae):.4f}")
    print(f"   RMSE: {np.mean(fc_rmse):.4f}")

    print(f"\n2. HAR Classification (F1-Score):")
    print(classification_report(y_true, y_pred))

🚀 Splitting by Participant: 120 Train / 31 Test
   Loading 120 files for train...
   Loading 31 files for test...

--- Starting Multi-Task Training ---
Epoch 1 Loss: 0.6024
Epoch 2 Loss: 0.5768
Epoch 3 Loss: 0.5689

--- Final Evaluation ---
1. Forecasting (1.0s Horizon):
   MAE: 0.0946
   RMSE: 0.1656

2. HAR Classification (F1-Score):
              precision    recall  f1-score   support

           0       0.58      0.90      0.70     66549
           1       0.54      0.70      0.61     62883
           2       0.37      0.13      0.20     14322
           3       0.64      0.15      0.25     34708
           4       0.72      0.74      0.73      1666
           5       0.61      0.36      0.45      6344
           6       0.35      0.28      0.31     93149

    accuracy                           0.51    279621
   macro avg       0.54      0.47      0.47    279621
weighted avg       0.49      0.51      0.46    279621



In [ ]:
# Define the indices we want to evaluate (0 to 5)
known_indices = [0, 1, 2, 3, 4, 5]
known_names = CLASSES[:-1]

print("\nAccuracy on KNOWN activities (excluding 'Unknown'):")
print(classification_report(
    y_true_clean,
    y_pred_clean,
    labels=known_indices,    # <--- Force it to look only at these IDs
    target_names=known_names # <--- Match names to those IDs
))


Accuracy on KNOWN activities (excluding 'Unknown'):
              precision    recall  f1-score   support

       sleep       0.91      0.90      0.91     66549
   sit-stand       0.78      0.70      0.74     62883
     walking       0.48      0.13      0.21     14322
       mixed       0.80      0.15      0.26     34708
   bicycling       0.80      0.74      0.76      1666
     vehicle       0.75      0.36      0.49      6344

   micro avg       0.84      0.62      0.71    186472
   macro avg       0.75      0.50      0.56    186472
weighted avg       0.81      0.62      0.66    186472

